In [2]:
#Step 1: Library Import

import pandas as pd
import numpy as np

In [3]:
#Step 2: Create Sample Datasets & Save as CSV (Skip this if you already have files)

customers_data = {
    'customer_id': [101, 102, 103, 104, 105, 106],
    'customer_name': ['Rahul', 'Priya', 'Aman', 'Sneha', 'Vikash', 'Anjali'],
    'city': ['Mumbai', 'Delhi', 'Mumbai', 'Bangalore', 'Delhi', 'Chennai'],
    'region': ['West', 'North', 'West', 'South', 'North', 'South']
}

orders_data = {
    'order_id': [1, 2, 3, 4, 5, 6, 7, 8],
    'customer_id': [101, 101, 102, 103, 103, 103, 105, 105],
    'order_amount': [5000, 7000, 3000, 4500, 2500, 6000, 8000, 2000]
}

pd.DataFrame(customers_data).to_csv('customers.csv', index=False)
pd.DataFrame(orders_data).to_csv('orders.csv', index=False)
print("customers.csv and orders.csv created successfully!")

customers.csv and orders.csv created successfully!


In [4]:
#Step 3: Data Ingestion

customers = pd.read_csv('customers.csv')
orders = pd.read_csv('orders.csv')
print("Customers Shape:", customers.shape)
print("Orders Shape:", orders.shape)
display(customers.head())
display(orders.head())

Customers Shape: (6, 4)
Orders Shape: (8, 3)


,customer_id,customer_name,city,region
0,101,Rahul,Mumbai,West
1,102,Priya,Delhi,North
2,103,Aman,Mumbai,West
3,104,Sneha,Bangalore,South
4,105,Vikash,Delhi,North


,order_id,customer_id,order_amount
0,1,101,5000
1,2,101,7000
2,3,102,3000
3,4,103,4500
4,5,103,2500


In [5]:
#Step 4: Relational Table Joins with pd.merge() on customer_id

#4a. INNER JOIN - Only customers with orders
merged_inner = pd.merge(customers, orders, on='customer_id', how='inner')
print("Inner Join Shape (only buyers):", merged_inner.shape)
display(merged_inner.head())

#4b. LEFT JOIN - All customers including non-buyers
merged_left = pd.merge(customers, orders, on='customer_id', how='left')
print("Left Join Shape (all customers):", merged_left.shape)
display(merged_left.head())

Inner Join Shape (only buyers): (8, 6)


,customer_id,customer_name,city,region,order_id,order_amount
0,101,Rahul,Mumbai,West,1,5000
1,101,Rahul,Mumbai,West,2,7000
2,102,Priya,Delhi,North,3,3000
3,103,Aman,Mumbai,West,4,4500
4,103,Aman,Mumbai,West,5,2500


Left Join Shape (all customers): (10, 6)


,customer_id,customer_name,city,region,order_id,order_amount
0,101,Rahul,Mumbai,West,1.0,5000.0
1,101,Rahul,Mumbai,West,2.0,7000.0
2,102,Priya,Delhi,North,3.0,3000.0
3,103,Aman,Mumbai,West,4.0,4500.0
4,103,Aman,Mumbai,West,5.0,2500.0


In [6]:
#Step 5: Metric Aggregation & Segmentation with .groupby() and .agg()

#5a. Total spend & order count per customer
customer_kpi = merged_inner.groupby(['customer_id', 'customer_name', 'city']).agg(
    Total_Spend=('order_amount', 'sum'),
    Order_Count=('order_id', 'count'),
    Average_Order_Value=('order_amount', 'mean')
).reset_index()
print("Customer Level KPIs:")
display(customer_kpi)

#5b. Average Order Value (AOV) for the whole business
aov = merged_inner['order_amount'].mean()
print(f"Overall Average Order Value (AOV): {aov:.2f}")

#5c. Total volume of orders segmented by city/region
city_metrics = merged_inner.groupby('city').agg(
    Total_Orders=('order_id', 'count'),
    Total_Revenue=('order_amount', 'sum')
).reset_index()
print("City-wise Segmentation:")
display(city_metrics)

Customer Level KPIs:


,customer_id,customer_name,city,Total_Spend,Order_Count,Average_Order_Value
0,101,Rahul,Mumbai,12000,2,6000.000000
1,102,Priya,Delhi,3000,1,3000.000000
2,103,Aman,Mumbai,13000,3,4333.333333
3,105,Vikash,Delhi,10000,2,5000.000000


Overall Average Order Value (AOV): 4750.00
City-wise Segmentation:


,city,Total_Orders,Total_Revenue
0,Delhi,3,13000
1,Mumbai,5,25000


In [7]:
#Step 6: Cohort Filtering & Leaderboard Formulation

#6a. Filter out non-converting accounts (customers with zero purchases)
non_converters = merged_left[merged_left['order_id'].isnull()]
print("Non-converting Customers (0 purchases):")
display(non_converters[['customer_id', 'customer_name', 'city']])

#6b. Clean summary table - Top 5 highest-spending customer profiles
top5_customers = customer_kpi.sort_values(by='Total_Spend', ascending=False).head(5)
print("TOP 5 Highest-Spending Customers:")
display(top5_customers)

#Save for README
top5_customers.to_csv('top5_customers.csv', index=False)

Non-converting Customers (0 purchases):


,customer_id,customer_name,city
6,104,Sneha,Bangalore
9,106,Anjali,Chennai


TOP 5 Highest-Spending Customers:


,customer_id,customer_name,city,Total_Spend,Order_Count,Average_Order_Value
2,103,Aman,Mumbai,13000,3,4333.333333
0,101,Rahul,Mumbai,12000,2,6000.000000
3,105,Vikash,Delhi,10000,2,5000.000000
1,102,Priya,Delhi,3000,1,3000.000000
